In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Alternative Approaches Comparison

This section analyzes all experimental approaches tested to improve the baseline performance.

**Important Note**: The correct baseline is **48.15%** (after OCR artifact removal). A previous baseline showing 67.39% had incorrect score calculations and has been corrected. All scores use the proper token_set_ratio calculation method.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define experimental results based on identified files
# NOTE: Baseline uses CORRECTED scores (48.15%). Previous baseline (67.39%) had incorrect score calculation.
experiments = {
    'Baseline (OCR Clean)': {
        'file': '../data/processed/archive/qa_results_2025-12-26_18-28-51_20s.csv',
        'mean': 0.4815,
        'passing': 4,
        'category': 'Baseline'
    },
    'Hybrid w=0.95': {
        'file': '../data/processed/archive/qa_results_2025-12-26_17-03-29_21s_w95.csv',
        'mean': 0.4309,
        'passing': 4,
        'category': 'Hybrid Weight'
    },
    'Pure Vector': {
        'file': '../data/processed/archive/qa_results_2025-12-26_17-04-07_13s_vector.csv',
        'mean': 0.4426,
        'passing': 3,
        'category': 'Hybrid Weight'
    },
    'Rerank Top 15': {
        'file': '../data/processed/archive/qa_results_2025-12-26_17-06-48_23s_r15.csv',
        'mean': 0.4480,
        'passing': 5,
        'category': 'CrossEncoder'
    },
    'Rerank Top 20': {
        'file': '../data/processed/archive/qa_results_2025-12-26_17-07-32_23s_r20.csv',
        'mean': 0.4412,
        'passing': 4,
        'category': 'CrossEncoder'
    },
    'Rerank Top 25': {
        'file': '../data/processed/archive/qa_results_2025-12-26_17-08-47_26s_r25.csv',
        'mean': 0.4837,
        'passing': 5,
        'category': 'CrossEncoder'
    },
    'Semantic Sentence': {
        'file': '../data/processed/archive/qa_results_2025-12-26_17-12-59_67s_semantic.csv',
        'mean': 0.4846,
        'passing': 3,
        'category': 'Answer Generation'
    },
    'Before OCR Clean': {
        'file': '../data/processed/archive/qa_results_2025-12-26_18-00-13_20s.csv',
        'mean': 0.4378,
        'passing': 4,
        'category': 'Pre-Improvement'
    }
}

# Create comparison dataframe
comparison_data = []
for name, data in experiments.items():
    comparison_data.append({
        'Experiment': name,
        'Mean Similarity': data['mean'] * 100,
        'Passing (≥80%)': data['passing'],
        'Category': data['category'],
        'Change vs Baseline': (data['mean'] - 0.4815) * 100
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Mean Similarity', ascending=False)

print("\n" + "="*80)
print("ALTERNATIVE APPROACHES COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)
print(f"\nBaseline (OCR Cleaned): 48.15% mean, 4/40 passing")
print(f"Pre-OCR Cleaning: 43.78% mean, 4/40 passing")
print(f"OCR artifact removal improved quality by +4.37%")
print(f"\nNote: Previous baseline showing 67.39% had incorrect score calculation.")
print(f"Actual scores were recalculated and corrected to 48.15%.")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Mean Similarity Comparison
colors = ['green' if cat == 'Baseline' else 'blue' if cat == 'Pre-Improvement' else 'red' if change < -5 else 'orange' 
          for cat, change in zip(comparison_df['Category'], comparison_df['Change vs Baseline'])]

ax1 = axes[0]
bars = ax1.barh(range(len(comparison_df)), comparison_df['Mean Similarity'], color=colors, alpha=0.7)
ax1.set_yticks(range(len(comparison_df)))
ax1.set_yticklabels(comparison_df['Experiment'])
ax1.set_xlabel('Mean Similarity (%)')
ax1.set_title('Mean Similarity Score by Approach')
ax1.axvline(x=48.15, color='green', linestyle='--', linewidth=2, label='Baseline (48.15%)')
ax1.axvline(x=43.78, color='blue', linestyle=':', linewidth=2, label='Pre-OCR (43.78%)')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Add value labels
for i, (val, change) in enumerate(zip(comparison_df['Mean Similarity'], comparison_df['Change vs Baseline'])):
    if abs(change) < 0.1:
        label = f'{val:.1f}%'
    else:
        label = f'{val:.1f}% ({change:+.1f}%)'
    ax1.text(val + 0.5, i, label, va='center', fontsize=8)

# Plot 2: Passing Questions Comparison
ax2 = axes[1]
bars = ax2.barh(range(len(comparison_df)), comparison_df['Passing (≥80%)'], color=colors, alpha=0.7)
ax2.set_yticks(range(len(comparison_df)))
ax2.set_yticklabels(comparison_df['Experiment'])
ax2.set_xlabel('Questions Passing (score ≥ 0.8)')
ax2.set_title('Passing Questions by Approach')
ax2.axvline(x=4, color='green', linestyle='--', linewidth=2, label='Baseline (4/40)')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, val in enumerate(comparison_df['Passing (≥80%)']):
    ax2.text(val + 0.15, i, f'{int(val)}/40', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Category-wise analysis
fig, ax = plt.subplots(figsize=(12, 6))

categories = comparison_df[~comparison_df['Category'].isin(['Baseline', 'Pre-Improvement'])].groupby('Category')
cat_names = []
cat_means = []
cat_colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']

for i, (cat_name, group) in enumerate(categories):
    cat_names.append(cat_name)
    cat_means.append(group['Mean Similarity'].mean())

# Add baseline and pre-improvement
cat_names.insert(0, 'Pre-OCR Clean')
cat_means.insert(0, 43.78)
cat_names.insert(0, 'Baseline (OCR)')
cat_means.insert(0, 48.15)

bars = ax.bar(range(len(cat_names)), cat_means, 
               color=['green', 'blue'] + cat_colors[:len(cat_names)-2], alpha=0.7)

ax.set_xticks(range(len(cat_names)))
ax.set_xticklabels(cat_names, rotation=45, ha='right')
ax.set_ylabel('Average Mean Similarity (%)')
ax.set_title('Performance by Experiment Category')
ax.axhline(y=48.15, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Baseline')
ax.axhline(y=43.78, color='blue', linestyle=':', linewidth=2, alpha=0.5, label='Pre-OCR')
ax.grid(axis='y', alpha=0.3)
ax.legend()

# Add value labels
for i, val in enumerate(cat_means):
    ax.text(i, val + 0.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)
print("1. Baseline (OCR cleaned): 48.15% mean, 4/40 passing")
print("2. OCR artifact removal: +4.37% improvement (43.78% → 48.15%)")
print("3. Hybrid Weight tuning: -5.06% to -3.89% (regression)")
print("4. CrossEncoder pool expansion: -3.35% to -0.03% (mostly regression)")
print("5. Semantic sentence selection: +0.31% (minimal improvement, slower)")
print("6. Extractive approach with 800 chars is optimal for current system")
print("="*80)
print("\nIMPORTANT: Previous baseline showing 67.39% had incorrect score calculation.")
print("All scores were recalculated using correct similarity method.")

## Detailed Score Distribution Analysis

Compare score distributions across different experimental approaches to understand how they affected answer quality.

In [ ]:
# Load and compare score distributions for key experiments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

key_experiments = [
    ('Baseline (OCR Clean)', '../data/processed/archive/qa_results_2025-12-26_18-28-51_20s.csv'),
    ('Hybrid w=0.95', '../data/processed/archive/qa_results_2025-12-26_17-03-29_21s_w95.csv'),
    ('Rerank Top 15', '../data/processed/archive/qa_results_2025-12-26_17-06-48_23s_r15.csv'),
    ('Semantic Sentence', '../data/processed/archive/qa_results_2025-12-26_17-12-59_67s_semantic.csv')
]

for idx, (name, filepath) in enumerate(key_experiments):
    ax = axes[idx // 2, idx % 2]
    
    try:
        df_exp = pd.read_csv(filepath)
        
        # Calculate statistics
        mean_score = df_exp['score'].mean() * 100
        passing = (df_exp['score'] >= 0.8).sum()
        
        # Plot histogram
        ax.hist(df_exp['score'] * 100, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
        ax.axvline(x=80, color='red', linestyle='--', linewidth=2, label='Passing Threshold (80%)')
        ax.axvline(x=mean_score, color='green', linestyle='-', linewidth=2, label=f'Mean ({mean_score:.1f}%)')
        
        ax.set_xlabel('Similarity Score (%)')
        ax.set_ylabel('Number of Questions')
        ax.set_title(f'{name}\nMean: {mean_score:.1f}%, Passing: {passing}/40')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
    except Exception as e:
        ax.text(0.5, 0.5, f'Error loading:\n{name}\n{str(e)}', 
                ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

print("\nNote: All scores use corrected calculation method (token_set_ratio).")
print("Previous baseline (67.39%) used incorrect scoring - actual baseline is 48.15%.")

## Performance Profiling: Answer Generation Bottleneck Analysis

Let's profile the retriever's generate_answer function to identify the main performance bottleneck.

In [ ]:
# Install required packages
import sys
import subprocess
try:
    from elasticsearch import Elasticsearch
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'elasticsearch'])
    from elasticsearch import Elasticsearch

In [ ]:
import sys
import os
sys.path.insert(0, '../src')

from search.retriever import RulebookRetriever
import time
import re

# Initialize retriever
retriever = RulebookRetriever(use_reranker=True)

# Sample question
question = "How many cards does each player start with?"

# Get chunks
chunks = retriever.search(question, top_k=10, search_type="hybrid")

print(f"Retrieved {len(chunks)} chunks")
print(f"Total text size: {sum(len(c.get('text', '')) for c in chunks)} chars")

# Profile the current implementation
start = time.time()
answer = retriever.generate_answer(question, chunks, multi_chunk_synthesis=True)
end = time.time()

print(f"\n=== Current Implementation ===")
print(f"Time: {(end-start)*1000:.1f}ms")
print(f"Answer: {answer[:200]}...")

# Now let's break down the steps
q_emb_time = 0
sentence_split_time = 0
encoding_time = 0
similarity_time = 0
sorting_time = 0

start = time.time()
q_emb = retriever.model.encode(question, convert_to_tensor=True)
q_emb_time = time.time() - start

all_sentences = []
sentence_count = 0
for ch in chunks[:10]:
    text = ch.get("text") or ""
    if not text:
        continue
    
    start = time.time()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.split()) >= 4 and not s.isupper()]
    sentence_split_time += time.time() - start
    
    sentence_count += len(sentences)
    all_sentences.extend(sentences)

print(f"\n=== Performance Breakdown ===")
print(f"Question embedding: {q_emb_time*1000:.1f}ms")
print(f"Sentence splitting: {sentence_split_time*1000:.1f}ms")
print(f"Total sentences to encode: {sentence_count}")

# Test batch vs individual encoding
if all_sentences[:50]:  # Limit for testing
    # Individual encoding (like current implementation per chunk)
    start = time.time()
    for s in all_sentences[:20]:
        _ = retriever.model.encode(s, convert_to_tensor=True)
    individual_time = time.time() - start
    
    # Batch encoding
    start = time.time()
    _ = retriever.model.encode(all_sentences[:20], convert_to_tensor=True)
    batch_time = time.time() - start
    
    print(f"\nEncoding 20 sentences:")
    print(f"  Individual (20 calls): {individual_time*1000:.1f}ms")
    print(f"  Batch (1 call): {batch_time*1000:.1f}ms")
    print(f"  Speedup: {individual_time/batch_time:.1f}x")
    
print(f"\n=== Key Finding ===")
print(f"The current implementation already uses batch encoding per chunk.")
print(f"Main bottleneck: Encoding {sentence_count} sentences across {len(chunks)} chunks")
print(f"Estimated sentence encoding time: ~{sentence_count * 5:.0f}ms (at 5ms/sentence avg)")